In [1]:
import requests
import pandas as pd
import os
from dotenv import load_dotenv
from datetime import datetime

In [2]:
load_dotenv()

True

# Conditions

## Past

In [ ]:
import requests
import pandas as pd
import os
from dotenv import load_dotenv
from datetime import datetime

load_dotenv()

OWM_KEY = os.getenv("OPENWEATHER_API_KEY")
TOMTOM_KEY = os.getenv("TOMTOM_API_KEY")

# LA neighborhood anchor points
# we'll query each one to give neighborhood-level granularity
NEIGHBORHOODS = {
    "Downtown":        {"lat": 34.0407, "lon": -118.2468},
    "Hollywood":       {"lat": 34.0928, "lon": -118.3287},
    "Santa Monica":    {"lat": 34.0195, "lon": -118.4912},
    "Silver Lake":     {"lat": 34.0869, "lon": -118.2708},
    "Koreatown":       {"lat": 34.0584, "lon": -118.3027},
    "San Fernando Valley": {"lat": 34.1836, "lon": -118.4466},
    "Long Beach":      {"lat": 33.7701, "lon": -118.1937},
    "West Hollywood":  {"lat": 34.0900, "lon": -118.3617},
}

In [ ]:
# Cell 3 — current weather by neighborhood
def get_weather(lat, lon):
    url = "https://api.openweathermap.org/data/2.5/weather"
    params = {
        "lat": lat,
        "lon": lon,
        "appid": OWM_KEY,
        "units": "imperial"  # Fahrenheit
    }
    r = requests.get(url, params=params)
    data = r.json()
    return {
        "temp_f":      data["main"]["temp"],
        "feels_like":  data["main"]["feels_like"],
        "humidity":    data["main"]["humidity"],
        "description": data["weather"][0]["description"],
        "wind_mph":    data["wind"]["speed"],
        "rain":        data.get("rain", {}).get("1h", 0),  # mm in last hour
    }

weather_rows = []
for name, coords in NEIGHBORHOODS.items():
    w = get_weather(coords["lat"], coords["lon"])
    w["neighborhood"] = name
    w["timestamp"] = datetime.now()
    weather_rows.append(w)
    print(f"✓ {name}: {w['temp_f']}°F, {w['description']}")

df_weather = pd.DataFrame(weather_rows)
df_weather

In [ ]:
# Cell 4 — AQI by neighborhood (same OWM key)
def get_aqi(lat, lon):
    url = "http://api.openweathermap.org/data/2.5/air_pollution"
    params = {"lat": lat, "lon": lon, "appid": OWM_KEY}
    r = requests.get(url, params=params)
    data = r.json()
    components = data["list"][0]["components"]
    return {
        "aqi":  data["list"][0]["main"]["aqi"],  # 1=Good, 2=Fair, 3=Moderate, 4=Poor, 5=Very Poor
        "pm2_5": components["pm2_5"],
        "pm10":  components["pm10"],
        "o3":    components["o3"],
        "no2":   components["no2"],
    }

aqi_rows = []
for name, coords in NEIGHBORHOODS.items():
    a = get_aqi(coords["lat"], coords["lon"])
    a["neighborhood"] = name
    a["timestamp"] = datetime.now()
    aqi_rows.append(a)
    print(f"✓ {name}: AQI={a['aqi']} (1=Good→5=Very Poor), PM2.5={a['pm2_5']}")

df_aqi = pd.DataFrame(aqi_rows)
df_aqi

In [ ]:
# Cell 5 — traffic by neighborhood via TomTom flow API
def get_traffic(lat, lon):
    # TomTom flow segment data — queries traffic at a specific point
    url = f"https://api.tomtom.com/traffic/services/4/flowSegmentData/absolute/10/json"
    params = {
        "point": f"{lat},{lon}",
        "key": TOMTOM_KEY,
    }
    r = requests.get(url, params=params)
    data = r.json()
    flow = data.get("flowSegmentData", {})
    current_speed = flow.get("currentSpeed", None)
    free_flow_speed = flow.get("freeFlowSpeed", None)
    
    # congestion ratio: 1.0 = free flow, lower = more congested
    ratio = (current_speed / free_flow_speed) if current_speed and free_flow_speed else None
    return {
        "current_speed_mph": current_speed,
        "free_flow_speed_mph": free_flow_speed,
        "congestion_ratio": round(ratio, 3) if ratio else None,
        "confidence": flow.get("confidence", None),
    }

traffic_rows = []
for name, coords in NEIGHBORHOODS.items():
    t = get_traffic(coords["lat"], coords["lon"])
    t["neighborhood"] = name
    t["timestamp"] = datetime.now()
    traffic_rows.append(t)
    print(f"✓ {name}: {t['current_speed_mph']} mph (free flow: {t['free_flow_speed_mph']} mph) → ratio: {t['congestion_ratio']}")

df_traffic = pd.DataFrame(traffic_rows)
df_traffic

In [ ]:
# Cell 6 — fix: use a shared timestamp
from datetime import datetime

SNAPSHOT_TIME = datetime.now().replace(microsecond=0)  # single timestamp for all

# re-tag each dataframe with the same timestamp
df_weather["timestamp"] = SNAPSHOT_TIME
df_aqi["timestamp"] = SNAPSHOT_TIME
df_traffic["timestamp"] = SNAPSHOT_TIME

# now merge works
df_conditions = df_weather.merge(df_aqi, on=["neighborhood", "timestamp"]) \
                           .merge(df_traffic, on=["neighborhood", "timestamp"])

df_conditions.to_csv("la_conditions_snapshot.csv", index=False)
print(f"Saved snapshot with {len(df_conditions)} neighborhoods")
df_conditions

## Mock Data (Grabs Future)

In [ ]:
# Cell 7 — pull 5-day / 3-hour forecast for all neighborhoods (weather + conditions over time)
import requests
import pandas as pd
import time
from datetime import datetime
from dotenv import load_dotenv
import os

load_dotenv()
OWM_KEY = os.getenv("OPENWEATHER_API_KEY")

NEIGHBORHOODS = {
    "Downtown":            {"lat": 34.0407, "lon": -118.2468},
    "Hollywood":           {"lat": 34.0928, "lon": -118.3287},
    "Santa Monica":        {"lat": 34.0195, "lon": -118.4912},
    "Silver Lake":         {"lat": 34.0869, "lon": -118.2708},
    "Koreatown":           {"lat": 34.0584, "lon": -118.3027},
    "San Fernando Valley": {"lat": 34.1836, "lon": -118.4466},
    "Long Beach":          {"lat": 33.7701, "lon": -118.1937},
    "West Hollywood":      {"lat": 34.0900, "lon": -118.3617},
}

def get_forecast(lat, lon, neighborhood):
    url = "https://api.openweathermap.org/data/2.5/forecast"
    params = {
        "lat": lat,
        "lon": lon,
        "appid": OWM_KEY,
        "units": "imperial"
    }
    r = requests.get(url, params=params, timeout=10)
    data = r.json()

    rows = []
    for entry in data.get("list", []):
        rows.append({
            "neighborhood":  neighborhood,
            "timestamp":     datetime.fromtimestamp(entry["dt"]),
            "temp_f":        entry["main"]["temp"],
            "feels_like":    entry["main"]["feels_like"],
            "humidity":      entry["main"]["humidity"],
            "wind_mph":      entry["wind"]["speed"],
            "rain":          entry.get("rain", {}).get("3h", 0),
            "description":   entry["weather"][0]["description"],
            "hour":          datetime.fromtimestamp(entry["dt"]).hour,
            "day_of_week":   datetime.fromtimestamp(entry["dt"]).strftime("%A"),
        })
    return rows

forecast_rows = []
for name, coords in NEIGHBORHOODS.items():
    rows = get_forecast(coords["lat"], coords["lon"], name)
    forecast_rows.extend(rows)
    print(f"✓ {name}: {len(rows)} time points")
    time.sleep(0.5)

df_forecast = pd.DataFrame(forecast_rows)
print(f"\nTotal rows: {len(df_forecast)}")
print(f"Date range: {df_forecast['timestamp'].min()} → {df_forecast['timestamp'].max()}")
df_forecast.head()

In [ ]:
# Cell 8 — pull AQI forecast for all neighborhoods
def get_aqi_forecast(lat, lon, neighborhood):
    url = "http://api.openweathermap.org/data/2.5/air_pollution/forecast"
    params = {"lat": lat, "lon": lon, "appid": OWM_KEY}
    r = requests.get(url, params=params, timeout=10)
    data = r.json()

    rows = []
    for entry in data.get("list", []):
        components = entry["components"]
        rows.append({
            "neighborhood": neighborhood,
            "timestamp":    datetime.fromtimestamp(entry["dt"]),
            "aqi":          entry["main"]["aqi"],
            "pm2_5":        components["pm2_5"],
            "pm10":         components["pm10"],
            "o3":           components["o3"],
            "no2":          components["no2"],
        })
    return rows

aqi_forecast_rows = []
for name, coords in NEIGHBORHOODS.items():
    rows = get_aqi_forecast(coords["lat"], coords["lon"], name)
    aqi_forecast_rows.extend(rows)
    print(f"✓ {name}: {len(rows)} time points")
    time.sleep(0.5)

df_aqi_forecast = pd.DataFrame(aqi_forecast_rows)
print(f"\nTotal rows: {len(df_aqi_forecast)}")
df_aqi_forecast.head()

In [ ]:
# Cell — merge weather and AQI forecasts
df_full_forecast = df_forecast.merge(df_aqi_forecast, on=["neighborhood", "timestamp"])

df_full_forecast.to_csv("la_forecast_data.csv", index=False)
print(f"Saved {len(df_full_forecast)} rows across {df_full_forecast['neighborhood'].nunique()} neighborhoods")
print(f"Columns: {list(df_full_forecast.columns)}")
df_full_forecast.head()

# Restaurants

In [6]:
# Cell — Google Places API connectivity test
GOOGLE_KEY = os.getenv("GOOGLE_MAPS_API_KEY")

In [10]:
# Cell — expanded anchor points + cuisine type diversity
# Cell — updated TRAINING_CITIES_EXPANDED with NYC included
TRAINING_CITIES = {
    "San Francisco": [
        {"lat": 37.7749, "lon": -122.4194},  # Downtown
        {"lat": 37.7599, "lon": -122.4148},  # Mission
        {"lat": 37.7831, "lon": -122.4039},  # SoMa
        {"lat": 37.8024, "lon": -122.4058},  # North Beach
        {"lat": 37.7695, "lon": -122.4469},  # Castro
        {"lat": 37.7857, "lon": -122.4011},  # Tenderloin
        {"lat": 37.7929, "lon": -122.4668},  # Haight
        {"lat": 37.7234, "lon": -122.4481},  # Excelsior
        {"lat": 37.7415, "lon": -122.4215},  # Bernal Heights
        {"lat": 37.7986, "lon": -122.4375},  # Russian Hill
    ],
    "San Diego": [
        {"lat": 32.7157, "lon": -117.1611},  # Downtown
        {"lat": 32.7480, "lon": -117.1386},  # Mission Valley
        {"lat": 32.8328, "lon": -117.2713},  # La Jolla
        {"lat": 32.6859, "lon": -117.1831},  # Chula Vista
        {"lat": 32.7553, "lon": -117.2069},  # Old Town
        {"lat": 32.7303, "lon": -117.1800},  # Hillcrest
        {"lat": 32.8157, "lon": -117.1350},  # Kearny Mesa
        {"lat": 32.6783, "lon": -117.0900},  # National City
        {"lat": 32.7636, "lon": -117.0697},  # Santee
        {"lat": 32.8025, "lon": -117.2317},  # Pacific Beach
    ],
    "Chicago": [
        {"lat": 41.8827, "lon": -87.6233},   # Downtown/Loop
        {"lat": 41.9215, "lon": -87.6530},   # Lincoln Park
        {"lat": 41.8955, "lon": -87.6280},   # River North
        {"lat": 41.9483, "lon": -87.6556},   # Wrigleyville
        {"lat": 41.8649, "lon": -87.6477},   # Pilsen
        {"lat": 41.8986, "lon": -87.6390},   # West Town
        {"lat": 41.9742, "lon": -87.6685},   # Andersonville
        {"lat": 41.8500, "lon": -87.6290},   # Bridgeport
        {"lat": 41.9031, "lon": -87.6866},   # Wicker Park
        {"lat": 41.8756, "lon": -87.6270},   # Chinatown
    ],
    "New York": [
        {"lat": 40.7580, "lon": -73.9855},   # Midtown
        {"lat": 40.7282, "lon": -73.9942},   # Greenwich Village
        {"lat": 40.7614, "lon": -73.9776},   # Upper East Side
        {"lat": 40.7489, "lon": -73.9680},   # Gramercy
        {"lat": 40.7831, "lon": -73.9712},   # Upper West Side
        {"lat": 40.7265, "lon": -73.9815},   # Lower East Side
        {"lat": 40.6892, "lon": -73.9442},   # Crown Heights Brooklyn
        {"lat": 40.7081, "lon": -73.9571},   # Williamsburg
        {"lat": 40.7282, "lon": -73.7949},   # Flushing Queens
        {"lat": 40.6501, "lon": -73.9496},   # Flatbush Brooklyn
    ]
}

# cuisine types to query separately to get variety
# these are Google Places API includedTypes
CUISINE_TYPES = [
    "restaurant",
    "chinese_restaurant",
    "mexican_restaurant",
    "italian_restaurant",
    "japanese_restaurant",
    "american_restaurant",
    "thai_restaurant",
    "indian_restaurant",
    "mediterranean_restaurant",
    "vietnamese_restaurant",
]

In [12]:
# Cell — expanded pull with cuisine diversity
import time

def get_restaurants_google_typed(lat, lon, place_type, radius=1200, max_results=20):
    url = "https://places.googleapis.com/v1/places:searchNearby"
    headers = {
        "Content-Type": "application/json",
        "X-Goog-Api-Key": GOOGLE_KEY,
        "X-Goog-FieldMask": "places.id,places.displayName,places.rating,places.userRatingCount,places.priceLevel,places.location,places.primaryTypeDisplayName,places.formattedAddress,places.regularOpeningHours"
    }
    body = {
        "includedTypes": [place_type],
        "maxResultCount": max_results,
        "locationRestriction": {
            "circle": {
                "center": {"latitude": lat, "longitude": lon},
                "radius": float(radius)
            }
        }
    }
    try:
        r = requests.post(url, headers=headers, json=body, timeout=15)
        r.raise_for_status()
        return r.json().get("places", [])
    except Exception as e:
        print(f"    Error ({place_type}): {e}")
        return []

def parse_place(place, city):
    price_map = {
        "PRICE_LEVEL_FREE": 0,
        "PRICE_LEVEL_INEXPENSIVE": 1,
        "PRICE_LEVEL_MODERATE": 2,
        "PRICE_LEVEL_EXPENSIVE": 3,
        "PRICE_LEVEL_VERY_EXPENSIVE": 4
    }
    price_raw = place.get("priceLevel", "PRICE_LEVEL_UNSPECIFIED")
    return {
        "place_id":          place.get("id"),
        "name":              place.get("displayName", {}).get("text"),
        "address":           place.get("formattedAddress"),
        "lat":               place.get("location", {}).get("latitude"),
        "lon":               place.get("location", {}).get("longitude"),
        "rating":            place.get("rating"),
        "user_rating_count": place.get("userRatingCount"),
        "price_level":       price_map.get(price_raw, None),
        "cuisine":           place.get("primaryTypeDisplayName", {}).get("text"),
        "has_hours":         place.get("regularOpeningHours") is not None,
        "city":              city,
    }

In [16]:
# Cell — full pull across all 4 cities
all_rows = []
seen_ids = set()

for city, anchors in TRAINING_CITIES.items():
    print(f"\n📍 Pulling {city}...")
    city_count = 0

    for i, anchor in enumerate(anchors):
        for cuisine in CUISINE_TYPES:
            places = get_restaurants_google_typed(
                anchor["lat"], anchor["lon"], cuisine
            )
            for place in places:
                pid = place.get("id")
                if pid and pid not in seen_ids:
                    seen_ids.add(pid)
                    all_rows.append(parse_place(place, city))
                    city_count += 1
            time.sleep(0.3)

        print(f"  Anchor {i+1}/{len(anchors)} done — {city_count} unique so far")

    print(f"  ✓ {city} complete: {city_count} unique restaurants")

df_restaurants = pd.DataFrame(all_rows)
print(f"\nTotal: {len(df_restaurants)} restaurants across {df_restaurants['city'].nunique()} cities")
print(df_restaurants["city"].value_counts())


📍 Pulling San Francisco...
  Anchor 1/10 done — 148 unique so far
  Anchor 2/10 done — 270 unique so far
  Anchor 3/10 done — 426 unique so far
  Anchor 4/10 done — 546 unique so far
  Anchor 5/10 done — 606 unique so far
  Anchor 6/10 done — 624 unique so far
  Anchor 7/10 done — 680 unique so far
  Anchor 8/10 done — 740 unique so far
  Anchor 9/10 done — 808 unique so far
  Anchor 10/10 done — 904 unique so far
  ✓ San Francisco complete: 904 unique restaurants

📍 Pulling San Diego...
  Anchor 1/10 done — 119 unique so far
  Anchor 2/10 done — 193 unique so far
  Anchor 3/10 done — 226 unique so far
  Anchor 4/10 done — 261 unique so far
  Anchor 5/10 done — 329 unique so far
  Anchor 6/10 done — 360 unique so far
  Anchor 7/10 done — 370 unique so far
  Anchor 8/10 done — 429 unique so far
  Anchor 9/10 done — 478 unique so far
  Anchor 10/10 done — 512 unique so far
  ✓ San Diego complete: 512 unique restaurants

📍 Pulling Chicago...
  Anchor 1/10 done — 145 unique so far
  Ancho

In [20]:
# Cell — data cleaning
print("Before cleaning:", df_restaurants.shape)

# 1. drop rows with null ratings
df_restaurants = df_restaurants.dropna(subset=["rating", "user_rating_count"])

# 2. impute null price levels with median per city + cuisine combo
df_restaurants["price_level"] = df_restaurants.groupby(
    ["city", "cuisine"]
)["price_level"].transform(lambda x: x.fillna(x.median()))

# fallback: city median
df_restaurants["price_level"] = df_restaurants.groupby(
    "city"
)["price_level"].transform(lambda x: x.fillna(x.median()))

# 3. fix null cuisine
df_restaurants["cuisine"] = df_restaurants["cuisine"].fillna("Restaurant")

# 4. drop duplicates
df_restaurants = df_restaurants.drop_duplicates(subset="place_id")

# 5. reset index
df_restaurants = df_restaurants.reset_index(drop=True)

print("After cleaning:", df_restaurants.shape)
print("\nNull counts after cleaning:")
print(df_restaurants.isnull().sum())
print("\nRestaurants per city:")
print(df_restaurants["city"].value_counts())
print("\nRating distribution:")
print(df_restaurants["rating"].describe())

Before cleaning: (3372, 11)
After cleaning: (3343, 11)

Null counts after cleaning:
place_id             0
name                 0
address              0
lat                  0
lon                  0
rating               0
user_rating_count    0
price_level          0
cuisine              0
has_hours            0
city                 0
dtype: int64

Restaurants per city:
city
New York         1117
San Francisco     897
Chicago           826
San Diego         503
Name: count, dtype: int64

Rating distribution:
count    3343.000000
mean        4.369339
std         0.346180
min         1.700000
25%         4.200000
50%         4.400000
75%         4.600000
max         5.000000
Name: rating, dtype: float64


In [24]:
# Cell — save
df_restaurants.to_csv("google_restaurants_training_clean.csv", index=False)
print(f"Saved {len(df_restaurants)} restaurants")
df_restaurants.head()

Saved 3343 restaurants


,place_id,name,address,lat,lon,rating,user_rating_count,price_level,cuisine,has_hours,city
0,ChIJlYL0Wa-BhYARJi6qr49Ncv0,Dumpling Home,"298 Gough St, San Francisco, CA 94102, USA",37.775821,-122.422636,4.5,1596.0,2.0,Chinese Restaurant,True,San Francisco
1,ChIJO7u9q5-AhYARiSSXyWv9eJ8,Zuni Café,"1658 Market St, San Francisco, CA 94102, USA",37.773600,-122.421608,4.4,2581.0,3.0,Californian Restaurant,True,San Francisco
2,ChIJC0_LRSB-j4ARUaZGUrDpvDM,Burma Love,"211 Valencia St, San Francisco, CA 94103, USA",37.769628,-122.422128,4.5,2229.0,2.0,Asian Restaurant,True,San Francisco
3,ChIJbTB3bKKAhYAR8Yqc6dEIMbQ,Souvla,"517 Hayes St, San Francisco, CA 94102, USA",37.776527,-122.424991,4.5,2315.0,2.0,Greek Restaurant,True,San Francisco
4,ChIJU889QiB-j4ARzQofqaFrMJU,Zeitgeist,"199 Valencia St, San Francisco, CA 94103, USA",37.770023,-122.422117,4.3,3098.0,1.0,Beer Garden,True,San Francisco
